In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
FEATURE_PATH = "cohort_feature_matrix_labeled_outcome_with_latent_class.csv"
df = pd.read_csv(FEATURE_PATH)

In [3]:
df

,subject_id,stay_id,hadm_id,anchor_age,gender,race,first_careunit,intime,outtime,charttime_hour,...,renal,respiration_24hours,coagulation_24hours,liver_24hours,cardiovascular_24hours,cns_24hours,renal_24hours,sofa_24hours,sepsis,latent_class
0,10000032,39553978,29079034,52,F,WHITE,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,2180-07-23 14:00:00,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,2.0,0,2.0
1,10000032,39553978,29079034,52,F,WHITE,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,2180-07-23 15:00:00,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,2.0,0,2.0
2,10000032,39553978,29079034,52,F,WHITE,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,2180-07-23 16:00:00,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,2.0,0,2.0
3,10000032,39553978,29079034,52,F,WHITE,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,2180-07-23 17:00:00,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,2.0,0,2.0
4,10000032,39553978,29079034,52,F,WHITE,Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,2180-07-23 18:00:00,...,NaN,0.0,0.0,0.0,1.0,1.0,0.0,2.0,0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2079748,19999987,36195440,23865745,57,F,UNKNOWN,Trauma SICU (TSICU),2145-11-02 22:59:00,2145-11-04 21:29:30,2145-11-04 17:00:00,...,NaN,2.0,1.0,0.0,1.0,2.0,1.0,7.0,0,6.0
2079749,19999987,36195440,23865745,57,F,UNKNOWN,Trauma SICU (TSICU),2145-11-02 22:59:00,2145-11-04 21:29:30,2145-11-04 18:00:00,...,NaN,2.0,1.0,0.0,1.0,2.0,1.0,7.0,0,6.0
2079750,19999987,36195440,23865745,57,F,UNKNOWN,Trauma SICU (TSICU),2145-11-02 22:59:00,2145-11-04 21:29:30,2145-11-04 19:00:00,...,NaN,2.0,1.0,0.0,1.0,2.0,1.0,7.0,0,6.0
2079751,19999987,36195440,23865745,57,F,UNKNOWN,Trauma SICU (TSICU),2145-11-02 22:59:00,2145-11-04 21:29:30,2145-11-04 20:00:00,...,0.0,2.0,1.0,0.0,1.0,2.0,1.0,7.0,0,6.0


In [4]:
# Parse timestamps if present
for c in ["intime", "outtime", "charttime_hour"]:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce")

# Create integer hour if needed
if "hour" not in df.columns:
    if {"charttime_hour", "intime"}.issubset(df.columns):
        df["hour"] = ((df["charttime_hour"] - df["intime"]).dt.total_seconds() / 3600.0)
        df["hour"] = np.floor(df["hour"]).astype("Int64")
    else:
        raise ValueError("Need either 'hour' OR both 'charttime_hour' and 'intime'.")

# Basic cleanup
df = df.dropna(subset=["stay_id", "hour"]).copy()
df["hour"] = df["hour"].astype(int)

KEYS = ["subject_id", "hadm_id", "stay_id", "hour"]
print(df[KEYS].head())


   subject_id   hadm_id   stay_id  hour
0    10000032  29079034  39553978     0
1    10000032  29079034  39553978     1
2    10000032  29079034  39553978     2
3    10000032  29079034  39553978     3
4    10000032  29079034  39553978     4


In [5]:
def to01(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.astype(int)
    ss = s.astype(str).str.strip().str.lower()
    mapping = {"true":1, "false":0, "1":1, "0":0, "yes":1, "no":0, "t":1, "f":0}
    out = ss.map(mapping)
    out_num = pd.to_numeric(s, errors="coerce")
    out = out.fillna(out_num)
    return out.astype(float).fillna(0).astype(int)

TARGET_COL = "sepsis" 
df[TARGET_COL] = to01(df[TARGET_COL])
print(df[TARGET_COL].value_counts(dropna=False))


sepsis
1    1482513
0     597240
Name: count, dtype: int64


In [6]:
drop_cols = set(KEYS + [TARGET_COL])

for c in ["intime", "outtime", "charttime_hour"]:
    if c in df.columns:
        drop_cols.add(c)

feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df[TARGET_COL].astype(int).copy()

print("X shape:", X.shape, "| y mean:", y.mean())

X shape: (2079753, 53) | y mean: 0.7128312833302801


In [7]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
groups = df["stay_id"].values

train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print("y_train:", y_train.value_counts().to_dict())
print("y_test :", y_test.value_counts().to_dict())

y_train: {1: 1186272, 0: 475545}
y_test : {1: 296241, 0: 121695}


In [8]:
# Drop datetime cols if any accidentally remain
dt_cols = X_train.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns.tolist()
if dt_cols:
    print("Dropping datetime feature cols:", dt_cols)
    X_train = X_train.drop(columns=dt_cols)
    X_test  = X_test.drop(columns=dt_cols)

cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ],
    remainder="drop"
)

model = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced_subsample"
    ))
])

model.fit(X_train, y_train)
print("Done training.")


Done training.


In [9]:
proba_test = model.predict_proba(X_test)[:, 1]

if y_test.nunique() == 2:
    print("ROC-AUC:", roc_auc_score(y_test, proba_test))
    print("PR-AUC :", average_precision_score(y_test, proba_test))
else:
    print("⚠️ y_test has one class; AUC not defined.")

ROC-AUC: 0.8519472980171363
PR-AUC : 0.9292068459215769


In [10]:
df.loc[X_test.index, "pred_proba_sepsis"] = proba_test

# optional: predicted label at threshold
thr = 0.5
df.loc[X_test.index, "pred_sepsis"] = (df.loc[X_test.index, "pred_proba_sepsis"] >= thr).astype(int)

out_path = "predictions_with_latent_class.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: predictions_with_latent_class.csv


In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

# ---- predicted probabilities aligned to X_test
p_hat = model.predict_proba(X_test)[:, 1]

# ---- ground truth aligned to X_test
if "y_test" in globals():
    y_true = np.asarray(y_test).astype(int)
else:
    # change df_feat -> whatever your full dataframe variable is called
    y_true = df_feat.loc[X_test.index, "sepsis"].astype(int).values

# ---- optional: store predictions back into your full df for inspection
# change df_feat -> your dataframe name
df.loc[X_test.index, "pred_proba_sepsis"] = p_hat


In [13]:
def eval_at_threshold(y_true, p_hat, thr):
    y_pred = (p_hat >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan   # recall / TPR
    specificity = tn / (tn + fp) if (tn + fp) else np.nan   # TNR
    precision   = tp / (tp + fp) if (tp + fp) else np.nan   # PPV
    npv         = tn / (tn + fn) if (tn + fn) else np.nan   # NPV

    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan
    bal_acc = (sensitivity + specificity) / 2 if (np.isfinite(sensitivity) and np.isfinite(specificity)) else np.nan
    f1 = (2 * precision * sensitivity / (precision + sensitivity)) if (precision + sensitivity) else np.nan

    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    fnr = fn / (fn + tp) if (fn + tp) else np.nan

    return {
        "threshold": float(thr),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "accuracy": acc,
        "balanced_acc": bal_acc,
        "precision": precision,
        "recall/sensitivity": sensitivity,
        "specificity": specificity,
        "NPV": npv,
        "F1": f1,
        "FPR": fpr,
        "FNR": fnr
    }


In [14]:
summary = {}
if len(np.unique(y_true)) == 2:
    summary["ROC-AUC"] = roc_auc_score(y_true, p_hat)
    summary["PR-AUC"]  = average_precision_score(y_true, p_hat)
else:
    summary["ROC-AUC"] = np.nan
    summary["PR-AUC"]  = np.nan

summary_df = pd.DataFrame([summary])
print("=== Threshold-free summary (test set) ===")
display(summary_df.style.format("{:.4f}"))


=== Threshold-free summary (test set) ===


,ROC-AUC,PR-AUC
0,0.8519,0.9292


In [15]:
threshold_list = [0.1, 0.2, 0.3, 0.5]
rows = [eval_at_threshold(y_true, p_hat, t) for t in threshold_list]
metrics_df = pd.DataFrame(rows)

print("=== Metrics at selected thresholds (test set) ===")
display(
    metrics_df[[
        "threshold","TP","FP","TN","FN",
        "precision","recall/sensitivity","specificity","F1",
        "accuracy","balanced_acc","NPV","FPR","FNR"
    ]].style.format({
        "precision":"{:.3f}",
        "recall/sensitivity":"{:.3f}",
        "specificity":"{:.3f}",
        "F1":"{:.3f}",
        "accuracy":"{:.3f}",
        "balanced_acc":"{:.3f}",
        "NPV":"{:.3f}",
        "FPR":"{:.3f}",
        "FNR":"{:.3f}",
    })
)


=== Metrics at selected thresholds (test set) ===


,threshold,TP,FP,TN,FN,precision,recall/sensitivity,specificity,F1,accuracy,balanced_acc,NPV,FPR,FNR
0,0.100000,294676,106279,15416,1565,0.735,0.995,0.127,0.845,0.742,0.561,0.908,0.873,0.005
1,0.200000,289040,86856,34839,7201,0.769,0.976,0.286,0.860,0.775,0.631,0.829,0.714,0.024
2,0.300000,279290,68835,52860,16951,0.802,0.943,0.434,0.867,0.795,0.689,0.757,0.566,0.057
3,0.500000,247199,38786,82909,49042,0.864,0.834,0.681,0.849,0.790,0.758,0.628,0.319,0.166


In [16]:
thresholds = np.linspace(0.01, 0.99, 99)
sweep = pd.DataFrame([eval_at_threshold(y_true, p_hat, t) for t in thresholds])

# Best F1 threshold
best_f1 = sweep.iloc[sweep["F1"].idxmax()][[
    "threshold","F1","precision","recall/sensitivity","specificity","TP","FP","TN","FN"
]]

# Best specificity with recall >= target
target_recall = 0.80
eligible = sweep[sweep["recall/sensitivity"] >= target_recall]
best_spec = None
if len(eligible) > 0:
    best_spec = eligible.iloc[eligible["specificity"].idxmax()][[
        "threshold","specificity","recall/sensitivity","precision","F1","TP","FP","TN","FN"
    ]]

print("=== Best operating points (test set) ===")
best_tbl = pd.DataFrame([best_f1])
best_tbl["criterion"] = "Best F1"
best_tbl = best_tbl[["criterion"] + [c for c in best_tbl.columns if c != "criterion"]]

if best_spec is not None:
    tmp = pd.DataFrame([best_spec])
    tmp["criterion"] = f"Max specificity with recall ≥ {target_recall}"
    tmp = tmp[["criterion"] + [c for c in tmp.columns if c != "criterion"]]
    best_tbl = pd.concat([best_tbl, tmp], ignore_index=True)

display(
    best_tbl.style.format({
        "threshold":"{:.2f}",
        "F1":"{:.3f}",
        "precision":"{:.3f}",
        "recall/sensitivity":"{:.3f}",
        "specificity":"{:.3f}",
    })
)

print("=== Top 10 thresholds by F1 ===")
display(
    sweep.sort_values("F1", ascending=False).head(10)[
        ["threshold","F1","precision","recall/sensitivity","specificity","TP","FP","TN","FN"]
    ].style.format({
        "threshold":"{:.2f}",
        "F1":"{:.3f}",
        "precision":"{:.3f}",
        "recall/sensitivity":"{:.3f}",
        "specificity":"{:.3f}",
    })
)


=== Best operating points (test set) ===


,criterion,threshold,F1,precision,recall/sensitivity,specificity,TP,FP,TN,FN
0,Best F1,0.34,0.868,0.815,0.928,0.488,274925.000000,62368.000000,59327.000000,21316.000000
1,Max specificity with recall ≥ 0.8,0.54,0.839,0.877,0.804,0.726,238131.000000,33326.000000,88369.000000,58110.000000


=== Top 10 thresholds by F1 ===


,threshold,F1,precision,recall/sensitivity,specificity,TP,FP,TN,FN
33,0.34,0.868,0.815,0.928,0.488,274925,62368,59327,21316
34,0.35,0.868,0.818,0.924,0.500,273662,60861,60834,22579
32,0.33,0.868,0.812,0.932,0.474,276035,63984,57711,20206
31,0.32,0.868,0.809,0.936,0.461,277208,65621,56074,19033
35,0.36,0.867,0.822,0.919,0.514,272217,59133,62562,24024
36,0.37,0.867,0.825,0.914,0.527,270897,57580,64115,25344
30,0.31,0.867,0.805,0.939,0.448,278263,67225,54470,17978
29,0.30,0.867,0.802,0.943,0.434,279290,68835,52860,16951
37,0.38,0.867,0.828,0.909,0.540,269428,56029,65666,26813
26,0.27,0.866,0.793,0.955,0.392,283053,74050,47645,13188
